## Clone Project

In [ ]:
%cd /content
!git clone https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/
!ls

## Installation

In [ ]:
%cd /content/Segment-Any-Anomaly

import re, pathlib
for p in pathlib.Path('GroundingDINO').rglob('*.py'):
    txt = p.read_text()
    patched = re.sub(r'transformers[^"\']*<4\.\d+\.\d+', 'transformers>=4.41.0', txt)
    if patched != txt:
        p.write_text(patched)

# GroundingDINO requires --no-build-isolation when torch is already installed (Colab default)
%cd GroundingDINO/
!pip install -q -e . --no-build-isolation
%cd ../SAM
!pip install -q -e .
!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru
%cd ..

In [ ]:
# Restart runtime to reload updated packages.
# After restart, Colab resets cwd to /content — subsequent cells handle this explicitly.
import os
os.kill(os.getpid(), 9)

## Download Weights

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd ./weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

## 🏃 Run Segment-Any-Anomaly Demo

In [ ]:
%cd /content/Segment-Any-Anomaly

import sys
sys.path.append('./GroundingDINO')
sys.path.append('./SAM')
sys.path.append('.')
import matplotlib.pyplot as plt
import SAA as SegmentAnyAnomaly
from utils.training_utils import *
import os
%matplotlib inline

## Preparation

In [ ]:
gpu_id = 0

os.environ['CURL_CA_BUNDLE'] = ''
os.environ['CUDA_VISIBLE_DEVICES'] = f"{gpu_id}"


dino_config_file = 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
dino_checkpoint = 'weights/groundingdino_swint_ogc.pth'
sam_checkpoint = 'weights/sam_vit_h_4b8939.pth'
box_threshold = 0.1
text_threshold = 0.1
eval_resolution = 1024
device = f"cuda:0"
root_dir = 'result'

# get the model
model = SegmentAnyAnomaly.Model(
    dino_config_file=dino_config_file,
    dino_checkpoint=dino_checkpoint,
    sam_checkpoint=sam_checkpoint,
    box_threshold=box_threshold,
    text_threshold=text_threshold,
    out_size=eval_resolution,
    device=device,
)

model = model.to(device)


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

# image_path = 'assets/candle.JPG'
# textual_prompts = ['color defect. hole. black defect. wick hole. spot. ', 'candle'] # detect prompts, filtered phrase
# property_text_prompts = 'the image of candle have 4 similar candle, with a maximum of 1 anomaly. The anomaly would not exceed 0.3 object area. '

# image_path = 'assets/carpet.png'
# textual_prompts = ['defect. ', 'carpet'] # detect prompts, filtered phrase
# property_text_prompts = 'the image of carpet have 1 dissimilar carpet, with a maximum of 5 anomaly. The anomaly would not exceed 0.9 object area. '

image_path = 'assets/capsule.JPG'
textual_prompts = [
        ['black melt. dark liquid.', 'capsules'],
        ['bubble', 'capsules'],  # 33+-->37+
    ] # detect prompts, filtered phrase
property_text_prompts = 'the image of capsule have 20 dissimilar capsule, with a maximum of 1 anomaly. The anomaly would not exceed 1. object area. '


image = cv2.imread(image_path)
cv2_imshow(image)

## Inference

In [ ]:
model.set_ensemble_text_prompts(textual_prompts, verbose=False)
model.set_property_text_prompts(property_text_prompts, verbose=False)


score, appendix = model(image)

similarity_map = appendix['similarity_map']

image_show = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_show = cv2.resize(image_show, (eval_resolution, eval_resolution))
similarity_map = cv2.resize(similarity_map, (eval_resolution, eval_resolution))
score = cv2.resize(score, (eval_resolution, eval_resolution))

plt.subplot(121)
plt.imshow(image_show)
plt.imshow(score, alpha=0.4,cmap='jet')
plt.title('Anomaly Score')

plt.subplot(122)
plt.imshow(image_show)
plt.imshow(similarity_map, alpha=0.4, cmap='jet')
plt.title('Saliency')
plt.show()